# Projet 5 — Machine Learning pour l'Allocation de Portefeuille

**Cours :** AI In Finance  
**Encadrants :** Nicolas De Roux & Mohamed El Fakir  
**Groupe :** [Nom du groupe]  
**Membres :**  
- Étudiant 1  
- Étudiant 2  
- Étudiant 3  

**Date :** Avril 2026

---

## Objectif du projet

L'objectif est d'utiliser des méthodes de Machine Learning pour prédire les rendements d'actions,  
puis d'utiliser ces prédictions pour construire un portefeuille optimisé.  
On compare plusieurs modèles (régression linéaire, Random Forest, LSTM) et on évalue  
la performance du portefeuille avec le ratio de Sharpe, le drawdown maximum et le rendement cumulé.

---

## 1. Installation et Imports

In [ ]:
# Installation des librairies nécessaires
!pip install yfinance scikit-learn torch matplotlib seaborn --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import torch
import torch.nn as nn
import torch.optim as optim

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)

print("Imports OK")
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)

## 2. Collecte des données

On télécharge les prix historiques de 5 actions du S&P 500 avec `yfinance`.  
Période : 2015-01-01 à 2024-12-31.

In [ ]:
# Liste des tickers choisis (secteurs variés)
tickers = ['AAPL', 'MSFT', 'JNJ', 'JPM', 'XOM']

# Téléchargement des données
data = yf.download(
    tickers=tickers,
    start='2015-01-01',
    end='2025-01-01',
    auto_adjust=False
)

print("Shape des données brutes:", data.shape)
data.head()

In [ ]:
# On garde seulement les prix de clôture ajustés
prices = data['Adj Close'].copy()
prices.columns = tickers
prices = prices.dropna()

print("Shape des prix:", prices.shape)
print("Période:", prices.index[0].date(), "à", prices.index[-1].date())
prices.head()

## 3. Exploration des données (EDA)

In [ ]:
# Statistiques descriptives des prix
prices.describe()

In [ ]:
# Vérification des valeurs manquantes
print("Valeurs manquantes par colonne:")
print(prices.isna().sum())

In [ ]:
# Evolution des prix normalisés (base 100)
prix_normalises = prices / prices.iloc[0] * 100

plt.figure(figsize=(12, 6))
for col in prix_normalises.columns:
    plt.plot(prix_normalises.index, prix_normalises[col], label=col)
plt.title('Evolution des prix normalisés (base 100)')
plt.xlabel('Date')
plt.ylabel('Prix normalisé')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Calcul des rendements journaliers
returns = prices.pct_change().dropna()

print("Shape des rendements:", returns.shape)
returns.describe()

In [ ]:
# Distribution des rendements
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(returns.columns):
    axes[i].hist(returns[col], bins=50, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Distribution rendements {col}')
    axes[i].axvline(returns[col].mean(), color='red', linestyle='--', label='Moyenne')
    axes[i].legend()

# On cache le dernier subplot si il y en a un de trop
if len(returns.columns) < len(axes):
    axes[-1].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Matrice de corrélation des rendements
corr_matrix = returns.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Corrélation des rendements journaliers')
plt.tight_layout()
plt.show()

In [ ]:
# Volatilité glissante (rolling) sur 30 jours
volatilite_30j = returns.rolling(window=30).std() * np.sqrt(252)

plt.figure(figsize=(12, 5))
for col in volatilite_30j.columns:
    plt.plot(volatilite_30j.index, volatilite_30j[col], label=col, alpha=0.7)
plt.title('Volatilité annualisée glissante (30 jours)')
plt.xlabel('Date')
plt.ylabel('Volatilité')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Feature Engineering

On crée des features techniques pour chaque action :  
- Rendements passés (lags)  
- Moyennes mobiles (5j, 20j)  
- Volatilité glissante  
- RSI (Relative Strength Index)  

La variable cible est le rendement du jour suivant.

In [ ]:
def creer_features(prix_serie, nom_ticker):
    """
    Crée les features pour une seule action.
    Entrée : série de prix (pd.Series)
    Sortie : DataFrame avec features et target
    """
    df = pd.DataFrame(index=prix_serie.index)
    
    # Rendement journalier
    df['return'] = prix_serie.pct_change()
    
    # Lags des rendements (1 à 5 jours)
    for lag in range(1, 6):
        df[f'return_lag{lag}'] = df['return'].shift(lag)
    
    # Moyennes mobiles
    df['ma5'] = prix_serie.rolling(5).mean()
    df['ma20'] = prix_serie.rolling(20).mean()
    df['ma_ratio'] = df['ma5'] / df['ma20']  # signal de tendance
    
    # Volatilité glissante (10 jours)
    df['volatilite_10j'] = df['return'].rolling(10).std()
    
    # RSI simplifié (14 jours)
    delta = prix_serie.diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    perte = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / perte
    df['rsi'] = 100 - (100 / (1 + rs))
    
    # Volume de rendement cumulé sur 5 jours
    df['momentum_5j'] = prix_serie.pct_change(5)
    
    # Target : rendement du jour suivant
    df['target'] = df['return'].shift(-1)
    
    # On enlève les colonnes intermédiaires qu'on ne veut pas en feature
    df = df.drop(columns=['return'])
    
    # Supprimer les lignes avec NaN
    df = df.dropna()
    
    return df

# Test sur AAPL
features_aapl = creer_features(prices['AAPL'], 'AAPL')
print("Shape features AAPL:", features_aapl.shape)
features_aapl.head()

In [ ]:
# Créer les features pour toutes les actions
features_dict = {}
for ticker in tickers:
    features_dict[ticker] = creer_features(prices[ticker], ticker)
    print(f"{ticker}: {features_dict[ticker].shape[0]} observations, {features_dict[ticker].shape[1]-1} features")

## 5. Modélisation — Prédiction des rendements

Pour chaque action, on entraîne 3 modèles :  
1. **Régression Linéaire** (baseline)  
2. **Ridge Regression** (régularisation)  
3. **Random Forest** (non-linéaire)  

On utilise un split temporel (pas de shuffle) pour respecter l'ordre du temps.

In [ ]:
def entrainer_modeles(df_features, ticker_name):
    """
    Entraîne les 3 modèles sur un dataset de features.
    Split temporel : 80% train, 20% test.
    Retourne les prédictions et les métriques.
    """
    # Séparer features et target
    X = df_features.drop(columns=['target'])
    y = df_features['target']
    
    # Split temporel (on ne mélange PAS)
    taille_train = int(len(X) * 0.8)
    X_train = X.iloc[:taille_train]
    X_test = X.iloc[taille_train:]
    y_train = y.iloc[:taille_train]
    y_test = y.iloc[taille_train:]
    
    # Scaling
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)
    
    # Modèle 1 : Régression Linéaire
    lr = LinearRegression()
    lr.fit(X_train_sc, y_train)
    pred_lr = lr.predict(X_test_sc)
    
    # Modèle 2 : Ridge
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_sc, y_train)
    pred_ridge = ridge.predict(X_test_sc)
    
    # Modèle 3 : Random Forest
    rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
    rf.fit(X_train_sc, y_train)
    pred_rf = rf.predict(X_test_sc)
    
    # Métriques
    resultats = {}
    for nom, pred in [('LinearReg', pred_lr), ('Ridge', pred_ridge), ('RandomForest', pred_rf)]:
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        mae = mean_absolute_error(y_test, pred)
        r2 = r2_score(y_test, pred)
        resultats[nom] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    
    resultats_df = pd.DataFrame(resultats).T
    print(f"\n--- Résultats pour {ticker_name} ---")
    print(resultats_df.round(6))
    
    return {
        'predictions': {'LinearReg': pred_lr, 'Ridge': pred_ridge, 'RandomForest': pred_rf},
        'y_test': y_test,
        'X_test': X_test,
        'scaler': scaler,
        'modeles': {'LinearReg': lr, 'Ridge': ridge, 'RandomForest': rf},
        'resultats': resultats_df,
        'feature_names': X.columns.tolist()
    }

# Entraîner les modèles pour chaque action
resultats_tous = {}
for ticker in tickers:
    resultats_tous[ticker] = entrainer_modeles(features_dict[ticker], ticker)

In [ ]:
# Comparaison des RMSE par modèle et par action
rmse_comparison = pd.DataFrame()
for ticker in tickers:
    rmse_comparison[ticker] = resultats_tous[ticker]['resultats']['RMSE']

print("\nComparaison des RMSE :")
print(rmse_comparison.round(6))

# Graphique
rmse_comparison.plot(kind='bar', figsize=(10, 5))
plt.title('RMSE par modèle et par action')
plt.ylabel('RMSE')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance du Random Forest (exemple avec AAPL)
rf_aapl = resultats_tous['AAPL']['modeles']['RandomForest']
feature_names = resultats_tous['AAPL']['feature_names']

importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_aapl.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 5))
plt.barh(importance['feature'], importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance — Random Forest (AAPL)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Modèle LSTM

On entraîne un LSTM sur les features de AAPL pour comparer avec les modèles classiques.  
C'est un réseau récurrent qui peut capter des dépendances temporelles.

In [ ]:
# Préparation des données pour le LSTM
# On prend AAPL comme exemple

df_lstm = features_dict['AAPL'].copy()
X_lstm_all = df_lstm.drop(columns=['target']).values
y_lstm_all = df_lstm['target'].values

# Split temporel
taille_train = int(len(X_lstm_all) * 0.8)
X_train_lstm = X_lstm_all[:taille_train]
X_test_lstm = X_lstm_all[taille_train:]
y_train_lstm = y_lstm_all[:taille_train]
y_test_lstm = y_lstm_all[taille_train:]

# Scaling
scaler_lstm = StandardScaler()
X_train_lstm_sc = scaler_lstm.fit_transform(X_train_lstm)
X_test_lstm_sc = scaler_lstm.transform(X_test_lstm)

# Créer des séquences pour le LSTM
def create_sequences(X, y, window_size=10):
    Xs, ys = [], []
    for i in range(len(X) - window_size):
        Xs.append(X[i:i+window_size])
        ys.append(y[i+window_size])
    return np.array(Xs), np.array(ys)

window = 10
X_train_seq, y_train_seq = create_sequences(X_train_lstm_sc, y_train_lstm, window)
X_test_seq, y_test_seq = create_sequences(X_test_lstm_sc, y_test_lstm, window)

print("Shape X_train_seq:", X_train_seq.shape)  # (samples, window, features)
print("Shape y_train_seq:", y_train_seq.shape)

In [ ]:
# Conversion en tensors PyTorch
X_train_t = torch.tensor(X_train_seq, dtype=torch.float32)
y_train_t = torch.tensor(y_train_seq, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test_seq, dtype=torch.float32)
y_test_t = torch.tensor(y_test_seq, dtype=torch.float32).unsqueeze(1)

print("Tensors créés")
print("X_train_t:", X_train_t.shape)
print("y_train_t:", y_train_t.shape)

In [ ]:
# Définition du modèle LSTM
class LSTMRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=32):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
    
    def forward(self, x):
        lstm_out, (hn, cn) = self.lstm(x)
        # On prend le dernier hidden state
        out = self.fc(hn[-1])
        return out

input_dim = X_train_t.shape[2]  # nombre de features
model_lstm = LSTMRegressor(input_dim=input_dim, hidden_dim=32)
criterion = nn.MSELoss()
optimizer = optim.Adam(model_lstm.parameters(), lr=1e-3)

print(model_lstm)

In [ ]:
# Entraînement du LSTM
epochs = 50
batch_size = 64
train_losses = []
val_losses = []

for epoch in range(epochs):
    model_lstm.train()
    
    # Mélanger les indices
    idx = torch.randperm(X_train_t.size(0))
    X_shuffled = X_train_t[idx]
    y_shuffled = y_train_t[idx]
    
    epoch_loss = 0
    n_batches = 0
    
    for i in range(0, X_train_t.size(0), batch_size):
        xb = X_shuffled[i:i+batch_size]
        yb = y_shuffled[i:i+batch_size]
        
        preds = model_lstm(xb)
        loss = criterion(preds, yb)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    train_losses.append(epoch_loss / n_batches)
    
    # Validation
    model_lstm.eval()
    with torch.no_grad():
        val_preds = model_lstm(X_test_t)
        val_loss = criterion(val_preds, y_test_t)
        val_losses.append(val_loss.item())
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} — Train Loss: {train_losses[-1]:.6f} — Val Loss: {val_losses[-1]:.6f}")

print("Entraînement terminé")

In [ ]:
# Courbes de loss
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Courbe d\'apprentissage LSTM')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluation du LSTM
model_lstm.eval()
with torch.no_grad():
    pred_lstm = model_lstm(X_test_t).numpy().flatten()

rmse_lstm = np.sqrt(mean_squared_error(y_test_seq, pred_lstm))
mae_lstm = mean_absolute_error(y_test_seq, pred_lstm)
r2_lstm = r2_score(y_test_seq, pred_lstm)

print(f"\nRésultats LSTM (AAPL) :")
print(f"  RMSE : {rmse_lstm:.6f}")
print(f"  MAE  : {mae_lstm:.6f}")
print(f"  R2   : {r2_lstm:.6f}")

# Comparaison avec les autres modèles
print(f"\nComparaison RMSE pour AAPL :")
print(f"  Linear Reg  : {resultats_tous['AAPL']['resultats'].loc['LinearReg', 'RMSE']:.6f}")
print(f"  Ridge       : {resultats_tous['AAPL']['resultats'].loc['Ridge', 'RMSE']:.6f}")
print(f"  Random Forest: {resultats_tous['AAPL']['resultats'].loc['RandomForest', 'RMSE']:.6f}")
print(f"  LSTM        : {rmse_lstm:.6f}")

## 7. Construction du portefeuille

On utilise les prédictions du Random Forest pour construire un portefeuille.  
Stratégie simple : **allocation proportionnelle aux rendements prédits positifs**.  
Si le rendement prédit est négatif, on n'investit pas dans cette action.

In [ ]:
# On récupère les prédictions RF sur la période de test pour chaque action
# et on les aligne par date

predictions_rf = pd.DataFrame()
rendements_reels = pd.DataFrame()

for ticker in tickers:
    res = resultats_tous[ticker]
    pred = res['predictions']['RandomForest']
    y_true = res['y_test']
    
    predictions_rf[ticker] = pd.Series(pred, index=y_true.index)
    rendements_reels[ticker] = y_true

# Garder seulement les dates communes
dates_communes = predictions_rf.dropna().index.intersection(rendements_reels.dropna().index)
predictions_rf = predictions_rf.loc[dates_communes]
rendements_reels = rendements_reels.loc[dates_communes]

print("Période de test:", dates_communes[0].date(), "à", dates_communes[-1].date())
print("Nombre de jours:", len(dates_communes))

In [ ]:
# Stratégie 1 : Portefeuille ML (allocation basée sur les prédictions)
def calculer_poids_ml(predictions_jour):
    """
    Calcule les poids du portefeuille à partir des prédictions.
    On investit proportionnellement aux rendements prédits positifs.
    Si tout est négatif, on reste en cash (poids = 0).
    """
    pred_positives = predictions_jour.clip(lower=0)
    somme = pred_positives.sum()
    if somme > 0:
        return pred_positives / somme
    else:
        return pd.Series(0, index=predictions_jour.index)

# Calculer les poids chaque jour
poids_ml = predictions_rf.apply(calculer_poids_ml, axis=1)

# Rendement du portefeuille ML
rendement_portfolio_ml = (poids_ml * rendements_reels).sum(axis=1)

# Stratégie 2 : Benchmark — Equal Weight (1/N)
n_actions = len(tickers)
rendement_equal_weight = rendements_reels.mean(axis=1)

print("Poids ML (premiers jours) :")
print(poids_ml.head())

## 8. Évaluation du portefeuille

In [ ]:
# Fonctions d'évaluation du portefeuille

def sharpe_ratio(rendements, taux_sans_risque=0.02):
    """Calcule le ratio de Sharpe annualisé."""
    exces = rendements - taux_sans_risque / 252
    if exces.std() == 0:
        return 0
    return np.sqrt(252) * exces.mean() / exces.std()

def max_drawdown(rendements):
    """Calcule le drawdown maximum."""
    cumul = (1 + rendements).cumprod()
    pic = cumul.cummax()
    drawdown = (cumul - pic) / pic
    return drawdown.min()

def rendement_annualise(rendements):
    """Calcule le rendement annualisé."""
    cumul_total = (1 + rendements).prod()
    n_annees = len(rendements) / 252
    return cumul_total ** (1 / n_annees) - 1

In [ ]:
# Métriques pour les deux stratégies
metriques = pd.DataFrame({
    'Portefeuille ML': [
        rendement_annualise(rendement_portfolio_ml),
        rendement_portfolio_ml.std() * np.sqrt(252),
        sharpe_ratio(rendement_portfolio_ml),
        max_drawdown(rendement_portfolio_ml)
    ],
    'Equal Weight (1/N)': [
        rendement_annualise(rendement_equal_weight),
        rendement_equal_weight.std() * np.sqrt(252),
        sharpe_ratio(rendement_equal_weight),
        max_drawdown(rendement_equal_weight)
    ]
}, index=['Rendement annualisé', 'Volatilité annualisée', 'Ratio de Sharpe', 'Max Drawdown'])

print("\n=== Comparaison des portefeuilles ===")
print(metriques.round(4))

In [ ]:
# Rendement cumulé des deux stratégies
cumul_ml = (1 + rendement_portfolio_ml).cumprod()
cumul_ew = (1 + rendement_equal_weight).cumprod()

plt.figure(figsize=(12, 6))
plt.plot(cumul_ml.index, cumul_ml.values, label='Portefeuille ML', linewidth=2)
plt.plot(cumul_ew.index, cumul_ew.values, label='Equal Weight (1/N)', linewidth=2, linestyle='--')
plt.title('Rendement cumulé : Portefeuille ML vs Equal Weight')
plt.xlabel('Date')
plt.ylabel('Valeur du portefeuille (base 1)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Drawdown au cours du temps
def calculer_drawdown_serie(rendements):
    cumul = (1 + rendements).cumprod()
    pic = cumul.cummax()
    return (cumul - pic) / pic

dd_ml = calculer_drawdown_serie(rendement_portfolio_ml)
dd_ew = calculer_drawdown_serie(rendement_equal_weight)

plt.figure(figsize=(12, 4))
plt.fill_between(dd_ml.index, dd_ml.values, 0, alpha=0.4, label='Drawdown ML')
plt.fill_between(dd_ew.index, dd_ew.values, 0, alpha=0.4, label='Drawdown Equal Weight')
plt.title('Drawdown au cours du temps')
plt.xlabel('Date')
plt.ylabel('Drawdown')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Allocation moyenne du portefeuille ML
allocation_moyenne = poids_ml.mean()

plt.figure(figsize=(8, 5))
plt.bar(allocation_moyenne.index, allocation_moyenne.values)
plt.title('Allocation moyenne du portefeuille ML')
plt.ylabel('Poids moyen')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

print("\nAllocation moyenne :")
print(allocation_moyenne.round(4))

## 9. Discussion et Conclusion

### Ce qui a bien fonctionné
- Le feature engineering avec les indicateurs techniques (RSI, moyennes mobiles, lags) donne des résultats cohérents.
- Le Random Forest capture des relations non-linéaires que la régression linéaire ne peut pas modéliser.
- Le split temporel est essentiel pour ne pas biaiser les résultats.

### Ce qui n'a pas fonctionné
- Les R² sont très faibles (proches de 0 ou négatifs), ce qui est normal pour la prédiction de rendements financiers : le signal est très bruité.
- Le LSTM n'apporte pas forcément de gain significatif par rapport au Random Forest sur ce type de données.
- La stratégie de portefeuille est simpliste (pas de contraintes de turnover, de coûts de transaction).

### Améliorations possibles
- Ajouter des données alternatives (sentiment de news, données macro).
- Utiliser des contraintes de portefeuille plus réalistes (limites de poids, coûts de transaction).
- Tester d'autres modèles (XGBoost, Transformers pour séries temporelles).
- Faire de l'optimisation de portefeuille de type Markowitz avec les prédictions.

### Conclusion
Ce projet montre que le Machine Learning peut être utilisé pour guider l'allocation de portefeuille,  
mais les rendements financiers sont extrêmement difficiles à prédire.  
Comme le disait George Box : **"All models are wrong, but some are useful."**